# Localization Simulation

Benchmark four trackers on one distance/angle localization scenario: the classical **CEKF** (centralized) and **DEKF** (diffusion) extended Kalman filters, the learned **DKN** (Distributed KalmanNet), and an optional **GNN-RNN** baseline.

Everything is driven by a trained experiment's `run.log`, and every figure is written to `<experiment>/notebook_plots/`.

**Segments**
1. Setup & imports
2. Experiment selection
3. Scenario setup & visualization
4. Single-trial comparison
5. Monte Carlo study (noise sweep)
6. Diffusion-coefficient network
7. dt-mismatch robustness
8. Node-count generalization
9. Time-step generalization
10. Inference-latency comparison
11. Detailed multi-model comparison

## 1. Setup & Imports

Load libraries, add the repository root to `sys.path`, and import the localization helpers - system/observation models, the classical filters, the DKN and GNN-RNN builders, checkpoint-path utilities, and plotting functions - used throughout the notebook.

In [ ]:
import importlib
from copy import deepcopy
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "utils").exists() else NOTEBOOK_DIR.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

torch.set_default_dtype(torch.float32)

import experiments.localization_experiment as localization_experiment_module
import utils.LocalizationScenario as localization_scenario_module
import utils.BaselineModels as baseline_models_module
importlib.reload(localization_scenario_module)
importlib.reload(localization_experiment_module)
importlib.reload(baseline_models_module)

from experiments.graphkalmanprocess_hparams import LOCALIZATION_BASELINE
from experiments.localization_experiment import (
    build_dkn_model,
    build_gnn_rnn_model,
    derive_localization_noise,
    dkn_model_path,
    dkn_run_name,
    gnn_rnn_model_path,
    list_experiments,
    load_state_dict_checked,
    load_experiment_config,
    measurement_noise_for_nodes,
    plot_graph,
    plot_tracking_results,
    plot_trajectory_and_nodes,
    predict_gnn_rnn,
)
from utils.ClassicDistributedKalman import (
    centralized_extended_kalman_filter,
    diffusion_extended_kalman_filter_parallel_edge,
)
from utils.LocalizationScenario import (
    ConstantVelocityModel,
    DistanceAngleObservation,
    build_graph_data_for_dkn,
    create_distance_based_graph,
    experiment_time_steps,
    farthest_mismatch_dt_ratio,
    format_dt_ratio,
    generate_measurements,
    generate_node_positions,
    generate_trajectory,
    generate_trial_data,
    nearest_nominal_dt_ratio,
    position_error_from_dekf,
    position_error_from_state_sequence,
    sync_torch_device,
)
from utils.reproducibility import seed_everything


## 2. Experiment Selection

List the trained `experiment_*` folders under `save_root` and print each one's key settings. The next segment loads the chosen experiment; its `run.log` alone drives the whole comparison.

In [ ]:
base_config = deepcopy(LOCALIZATION_BASELINE)
save_root = REPO_ROOT / base_config["save_root"]

experiments = list_experiments(save_root)
if not experiments:
    raise RuntimeError(f"No experiment_* folders found under {save_root}")

print("Available experiments:")
for i, exp in enumerate(experiments, start=1):
    tag = ""
    description = ""
    try:
        ecfg = load_experiment_config(exp)
    except FileNotFoundError:
        ecfg = None
    if ecfg is not None:
        exp_title = ecfg.get("title", "")
        title_str = f' - "{exp_title}"' if exp_title else ""
        tag = f"  (T={experiment_time_steps(ecfg)}, lr={ecfg.get('learning_rate','?')}){title_str}"
        description = ecfg.get("description", "")
    print(f"  [{i}] {exp.name}{tag}")
    if description:
        print(f"      {description}")


## 3. Scenario Setup & Visualization

Load the selected experiment's config and rebuild the scenario it was trained on: constant-velocity dynamics, distance/angle sensors, and the sensor graph. Available DKN / GNN-RNN checkpoints are discovered, the output folder `notebook_plots/` is created, and the sensor graph plus a sample trajectory are plotted so the geometry is clear before benchmarking.

In [ ]:
EXPERIMENT = len(experiments)

if not 1 <= EXPERIMENT <= len(experiments):
    raise ValueError(f"EXPERIMENT must be in [1, {len(experiments)}], got {EXPERIMENT}")
experiment_dir = experiments[EXPERIMENT - 1]
config_val = load_experiment_config(experiment_dir)
print(f"Loaded experiment: {experiment_dir.name}")
print(pd.Series(config_val))

seed_everything(config_val["seed"])

num_nodes = config_val["num_nodes"]
use_dt_mismatch = config_val.get("use_dt_mismatch", False)
dt_mismatch_values = config_val.get("dt_mismatch_values", [1.0])
default_dkn_dt_ratio = nearest_nominal_dt_ratio(dt_mismatch_values) if use_dt_mismatch else None

state_dimension = config_val["state_dimension"]
x_init = np.array(config_val["x0"], dtype=float).reshape(state_dimension, 1)
p0 = np.eye(state_dimension) * config_val["p0_scale"]


def sample_initial_state(mean, covariance):
    """Sample the true initial state from N(mean, covariance)."""
    sample = np.random.multivariate_normal(mean[:, 0], covariance)
    return sample.reshape(mean.shape)


num_time_steps = experiment_time_steps(config_val)
num_trials = config_val["num_trials"]
time_delta = config_val["time_delta"]
measurement_noise_values = config_val["r_scale"]
node_positions = np.array(config_val["node_positions"], dtype=float)
f_system = ConstantVelocityModel(time_delta)
f_system_dkn = ConstantVelocityModel(time_delta * float(default_dkn_dt_ratio)) if use_dt_mismatch else f_system
h_system = DistanceAngleObservation(node_positions)
node_types = h_system.node_classification[:, 0].numpy().astype(int)

def noise_for_scale(r_scale):
    return derive_localization_noise(config_val["mu"], config_val["rho"], r_scale)

def r_array_for(observation_model, r_scale):
    noise = noise_for_scale(r_scale)
    return measurement_noise_for_nodes(observation_model, noise["sigma_r"], noise["sigma_theta"])

adjacency_matrix = create_distance_based_graph(
    node_positions,
    k_neighbors=config_val["k_neighbors"],
    seed=config_val["graph_seed"],
)
j_matrix = np.array(adjacency_matrix, dtype=float, copy=True)
np.fill_diagonal(j_matrix, 1.0)

dkn_dir = experiment_dir
gnn_rnn_dir = experiment_dir / "gnn-rnn"
available_noise_values = [
    r for r in measurement_noise_values
    if dkn_model_path(
        experiment_dir,
        r,
        use_dt_mismatch=use_dt_mismatch,
        default_dt_ratio=default_dkn_dt_ratio,
    ).exists()
]
missing_noise_values = [r for r in measurement_noise_values if r not in available_noise_values]
if missing_noise_values:
    print(f"Warning: missing DKN checkpoints for r={missing_noise_values} in {dkn_dir} - skipping.")
if not available_noise_values:
    raise RuntimeError(f"No DKN checkpoints found in {dkn_dir}")

gnn_rnn_available_noise_values = [r for r in available_noise_values if gnn_rnn_model_path(experiment_dir, r).exists()]
missing_gnn_rnn_noise_values = [r for r in available_noise_values if r not in gnn_rnn_available_noise_values]
if missing_gnn_rnn_noise_values:
    print(f"Warning: missing GNN-RNN checkpoints for r={missing_gnn_rnn_noise_values} in {gnn_rnn_dir} - omitting GNN-RNN for those rows.")
use_gnn_rnn = len(gnn_rnn_available_noise_values) > 0
measurement_noise_values = available_noise_values

notebook_plot_dir = experiment_dir / "notebook_plots"
notebook_plot_dir.mkdir(parents=True, exist_ok=True)

print(f"Train nodes:      {num_nodes}")
print(f"Time steps:       {num_time_steps}")
if use_dt_mismatch:
    print(f"DKN checkpoint dt ratio for non-mismatch sections: {default_dkn_dt_ratio}")
plot_graph(adjacency_matrix, node_positions, title="Localization Scenario Graph")
preview_noise = noise_for_scale(measurement_noise_values[0])
plot_trajectory_and_nodes(
    node_positions,
    node_types,
    generate_trajectory(f_system, x_init, num_time_steps, preview_noise["q_matrix"]),
)


## 4. Single-Trial Comparison

Run a single trajectory and overlay the estimates from CEKF, DEKF, DKN, and (when a checkpoint exists) GNN-RNN on identical measurements - a quick qualitative look at tracking quality before the statistical sweeps.

In [ ]:
r_demo = 0.25 if 0.25 in measurement_noise_values else measurement_noise_values[0]
demo_noise = noise_for_scale(r_demo)
r_array_demo = r_array_for(h_system, r_demo)
true_x0 = sample_initial_state(x_init, p0)
trajectory, measurements = generate_trial_data(
    f_system, h_system, true_x0, num_time_steps, demo_noise["q_matrix"], r_array_demo
)
graph_data = build_graph_data_for_dkn(adjacency_matrix, h_system, trajectory, measurements)

x_hat_cekf = centralized_extended_kalman_filter(
    measurements=measurements,
    f_system=f_system,
    h_system=h_system,
    r_array=r_array_demo,
    q=demo_noise["q"],
    p0=p0,
    x0=x_init,
    time_steps=num_time_steps,
    node_num=num_nodes,
    q_matrix=demo_noise["q_matrix"],
)
x_hat_dekf = diffusion_extended_kalman_filter_parallel_edge(
    measurements=measurements,
    f_system=f_system,
    h_system=h_system,
    r_array=r_array_demo,
    q=demo_noise["q"],
    p0=p0,
    x0=x_init,
    j_matrix=j_matrix,
    time_steps=num_time_steps,
    node_num=num_nodes,
    q_matrix=demo_noise["q_matrix"],
)

example_model_path = dkn_model_path(
    experiment_dir,
    r_demo,
    use_dt_mismatch=use_dt_mismatch,
    default_dt_ratio=default_dkn_dt_ratio,
)
kalman_process = build_dkn_model(config_val, f_system_dkn, r_array_demo, x_init)
kalman_process = load_state_dict_checked(kalman_process, example_model_path)
with torch.no_grad():
    x_hat_dkn = kalman_process(graph_data)[0].mean(dim=1)[..., 0].cpu().numpy()

x_hat_gnn_rnn = None
gnn_path_demo = gnn_rnn_model_path(experiment_dir, r_demo)
if gnn_path_demo.exists():
    gnn_rnn_process = build_gnn_rnn_model(config_val)
    gnn_rnn_process = load_state_dict_checked(gnn_rnn_process, gnn_path_demo)
    x_hat_gnn_rnn = predict_gnn_rnn(gnn_rnn_process, graph_data, h_system)

demo_tracking_plot_path = notebook_plot_dir / f"{dkn_run_name(r_demo, use_dt_mismatch=use_dt_mismatch, default_dt_ratio=default_dkn_dt_ratio)}_tracking_comparison.png"
plot_tracking_results(
    trajectory,
    x_hat_cekf,
    x_hat_dekf=x_hat_dekf,
    x_hat_dkn=x_hat_dkn,
    x_hat_gnn_rnn=x_hat_gnn_rnn,
    node_positions=node_positions,
    node_types=node_types,
    save_path=demo_tracking_plot_path,
)


## 5. Monte Carlo Study (Noise Sweep)

For each measurement-noise level, average the position error over paired random trials. CEKF, DEKF, and DKN are evaluated with matched and mismatched `dt` dynamics, while GNN-RNN (no explicit dynamics) is evaluated once. The notebook produces one noise-sweep figure: **position error in dB** against inverse noise variance **$1/r^2$ (dB)**.

In [ ]:
if use_dt_mismatch:
    MONTE_CARLO_NO_MISMATCH_RATIO = nearest_nominal_dt_ratio(dt_mismatch_values)
    MONTE_CARLO_MISMATCH_RATIO = farthest_mismatch_dt_ratio(
        dt_mismatch_values, MONTE_CARLO_NO_MISMATCH_RATIO
    )
    monte_carlo_dt_ratios = [
        MONTE_CARLO_NO_MISMATCH_RATIO,
        MONTE_CARLO_MISMATCH_RATIO,
    ]
    print(
        "Monte Carlo dt ratios: "
        f"no mismatch={format_dt_ratio(MONTE_CARLO_NO_MISMATCH_RATIO)}, "
        f"mismatch={format_dt_ratio(MONTE_CARLO_MISMATCH_RATIO)}"
    )
else:
    # dt_mismatch_values is ignored for matched runs; it may hold a stale ratio.
    MONTE_CARLO_NO_MISMATCH_RATIO = 1.0
    MONTE_CARLO_MISMATCH_RATIO = None
    monte_carlo_dt_ratios = [MONTE_CARLO_NO_MISMATCH_RATIO]
    print("No dt-mismatch experiment is configured; evaluating nominal dynamics only.")

monte_carlo_labels = {
    model_name: {
        ratio: f"{model_name} dt x{format_dt_ratio(ratio)}"
        for ratio in monte_carlo_dt_ratios
    }
    for model_name in ["CEKF", "DEKF", "DKN"]
}
monte_carlo_series = [
    monte_carlo_labels[model_name][ratio]
    for model_name in ["CEKF", "DEKF", "DKN"]
    for ratio in monte_carlo_dt_ratios
] + ["GNN-RNN"]

avg_errors = {label: [] for label in monte_carlo_series}

for r_noise in measurement_noise_values:
    noise = noise_for_scale(r_noise)
    r_array = r_array_for(h_system, r_noise)

    dkn_processes = {}
    for dt_ratio in monte_carlo_dt_ratios:
        model_path = dkn_model_path(experiment_dir, r_noise, use_dt_mismatch=use_dt_mismatch, dt_ratio=dt_ratio)
        if not model_path.exists():
            raise FileNotFoundError(f"Missing DKN checkpoint for Monte Carlo comparison: {model_path}")
        f_model_ratio = ConstantVelocityModel(time_delta * float(dt_ratio))
        kalman_process = build_dkn_model(config_val, f_model_ratio, r_array, x_init)
        dkn_processes[dt_ratio] = load_state_dict_checked(kalman_process, model_path)

    gnn_rnn_process = None
    gnn_path = gnn_rnn_model_path(experiment_dir, r_noise)
    if gnn_path.exists():
        gnn_rnn_process = build_gnn_rnn_model(config_val)
        gnn_rnn_process = load_state_dict_checked(gnn_rnn_process, gnn_path)
    else:
        print(f"No GNN-RNN checkpoint found for r={r_noise}; GNN-RNN result will be NaN.")

    trial_errors = {label: [] for label in monte_carlo_series}

    for _ in tqdm(range(num_trials), desc=f"r={r_noise}"):
        true_x0 = sample_initial_state(x_init, p0)
        trajectory, measurements = generate_trial_data(
            f_system, h_system, true_x0, num_time_steps, noise["q_matrix"], r_array
        )
        graph_data = build_graph_data_for_dkn(adjacency_matrix, h_system, trajectory, measurements)

        for dt_ratio in monte_carlo_dt_ratios:
            f_model_ratio = ConstantVelocityModel(time_delta * float(dt_ratio))
            x_hat_cekf = centralized_extended_kalman_filter(
                measurements=measurements,
                f_system=f_model_ratio,
                h_system=h_system,
                r_array=r_array,
                q=noise["q"],
                p0=p0,
                x0=x_init,
                time_steps=num_time_steps,
                node_num=num_nodes,
                q_matrix=noise["q_matrix"],
            )
            x_hat_dekf = diffusion_extended_kalman_filter_parallel_edge(
                measurements=measurements,
                f_system=f_model_ratio,
                h_system=h_system,
                r_array=r_array,
                q=noise["q"],
                p0=p0,
                x0=x_init,
                j_matrix=j_matrix,
                time_steps=num_time_steps,
                node_num=num_nodes,
                q_matrix=noise["q_matrix"],
            )
            with torch.no_grad():
                x_hat_dkn = dkn_processes[dt_ratio](graph_data)[0].mean(dim=1)[..., 0].cpu().numpy()

            trial_errors[monte_carlo_labels["CEKF"][dt_ratio]].append(position_error_from_state_sequence(trajectory, x_hat_cekf[:, :, 0]))
            trial_errors[monte_carlo_labels["DEKF"][dt_ratio]].append(position_error_from_dekf(trajectory, x_hat_dekf))
            trial_errors[monte_carlo_labels["DKN"][dt_ratio]].append(position_error_from_state_sequence(trajectory, x_hat_dkn))

        if gnn_rnn_process is not None:
            x_hat_gnn_rnn = predict_gnn_rnn(gnn_rnn_process, graph_data, h_system)
            trial_errors["GNN-RNN"].append(position_error_from_state_sequence(trajectory, x_hat_gnn_rnn))

    for label in monte_carlo_series:
        avg_errors[label].append(np.mean(trial_errors[label]) if trial_errors[label] else np.nan)

results_df = pd.DataFrame(
    {
        "measurement_noise": measurement_noise_values,
        **avg_errors,
    }
)
results_df


In [ ]:
summary_plot_path = notebook_plot_dir / f"position_error_vs_inverse_r2_trials={num_trials}_T={num_time_steps}.png"
model_markers = {"CEKF": "o", "DEKF": "s", "DKN": "^"}
monte_carlo_plot_styles = {
    monte_carlo_labels[model_name][ratio]: (
        f"{model_markers[model_name]}-"
        if ratio == MONTE_CARLO_NO_MISMATCH_RATIO
        else f"{model_markers[model_name]}--"
    )
    for model_name in ["CEKF", "DEKF", "DKN"]
    for ratio in monte_carlo_dt_ratios
}
monte_carlo_plot_styles["GNN-RNN"] = "d:"

inv_r2_db = 10 * np.log10(1.0 / results_df["measurement_noise"] ** 2)
plt.figure(figsize=(11, 7))
for label, style in monte_carlo_plot_styles.items():
    if label in results_df.columns and results_df[label].notna().any():
        plt.plot(
            inv_r2_db,
            10 * np.log10(results_df[label]),
            style,
            linewidth=2,
            markersize=7,
            label=label,
        )
plt.xlabel(r"$1/r^2$ [dB]")
plt.ylabel("Position Error [dB]")
plt.title(
    f"Localization Noise Sweep ({num_trials} trials, mu={config_val['mu']}, "
    f"T={num_time_steps})"
)
plt.grid(True, alpha=0.3)
plt.legend(ncol=2)
plt.savefig(summary_plot_path, dpi=200, bbox_inches="tight")
print(f"Saved position-error plot: {summary_plot_path}")
plt.show()

## 6. Diffusion-Coefficient Network

*Does the network of diffusion coefficients the DEKF uses to combine neighbors change the results?* Rebuild `j_matrix` on the same sensor graph using binary, normalized-uniform, Metropolis-Hastings, and inverse-distance weights.

The non-trivial schemes are scaled to a common row sum so their process-noise scaling matches. Results are presented as a table plus relative-spread statistics; the only position-error-vs-`1/r²` figure remains in the Monte Carlo section.

In [ ]:
DIFFUSION_NUM_TRIALS = num_trials
diff_f_system = ConstantVelocityModel(time_delta)   # true dynamics, no dt mismatch


def _sinkhorn_symmetric(weight, iters=200, eps=1e-12):
    """Scale a symmetric non-negative matrix to (approx.) doubly stochastic, preserving zeros."""
    w = np.array(weight, dtype=float, copy=True)
    for _ in range(iters):
        w = w / (w.sum(axis=1, keepdims=True) + eps)
        w = w / (w.sum(axis=0, keepdims=True) + eps)
    return 0.5 * (w + w.T)


def build_diffusion_coefficients(adjacency, node_positions, scheme):
    """Symmetric combination-weight matrix ('network of diffusion coefficients') on the
    fixed graph support (neighbors + self-loops)."""
    support = np.array(adjacency, dtype=float, copy=True)
    np.fill_diagonal(support, 0.0)
    neighbor_mask = support > 0
    degree = neighbor_mask.sum(axis=1)
    n = support.shape[0]
    rows, cols = np.where(neighbor_mask)

    if scheme == "current (binary)":
        return neighbor_mask.astype(float) + np.eye(n)
    if scheme == "uniform (normalized)":
        return _sinkhorn_symmetric(neighbor_mask.astype(float) + np.eye(n))
    if scheme == "metropolis":
        w = np.zeros((n, n))
        for i, j in zip(rows, cols):
            w[i, j] = 1.0 / (1.0 + max(degree[i], degree[j]))
        np.fill_diagonal(w, 1.0 - w.sum(axis=1))
        return w
    if scheme == "inverse-distance":
        w = np.zeros((n, n))
        for i, j in zip(rows, cols):
            w[i, j] = 1.0 / (1.0 + np.linalg.norm(node_positions[i] - node_positions[j]))
        np.fill_diagonal(w, 1.0)
        return _sinkhorn_symmetric(w)
    raise ValueError(f"Unknown scheme: {scheme}")


# The DEKF scales its process noise by the coefficient row-sum, so put every non-trivial
# scheme on a common row-sum => identical process-noise term (isolates the *shape* of the network).
_support = np.array(adjacency_matrix, dtype=float, copy=True)
np.fill_diagonal(_support, 0.0)
diffusion_target_scale = float(((_support > 0).sum(axis=1) + 1.0).mean())

diffusion_schemes = ["current (binary)", "uniform (normalized)", "metropolis", "inverse-distance"]
diffusion_j_matrices = {}
for scheme in diffusion_schemes:
    w = build_diffusion_coefficients(adjacency_matrix, node_positions, scheme)
    if scheme != "current (binary)":
        w = w * diffusion_target_scale
    diffusion_j_matrices[scheme] = w

diffusion_errors = {scheme: [] for scheme in diffusion_schemes}
diffusion_errors_cekf = []

for r_noise in measurement_noise_values:
    noise = noise_for_scale(r_noise)
    r_array = r_array_for(h_system, r_noise)
    trials = [
        generate_trial_data(
            diff_f_system,
            h_system,
            sample_initial_state(x_init, p0),
            num_time_steps,
            noise["q_matrix"],
            r_array,
        )
        for _ in range(DIFFUSION_NUM_TRIALS)
    ]
    cekf_errs = []
    scheme_errs = {scheme: [] for scheme in diffusion_schemes}
    for trajectory, measurements in tqdm(trials, desc=f"diffusion r={r_noise}"):
        x_hat_cekf = centralized_extended_kalman_filter(
            measurements=measurements, f_system=diff_f_system, h_system=h_system,
            r_array=r_array, q=noise["q"], p0=p0, x0=x_init,
            time_steps=num_time_steps, node_num=num_nodes,
            q_matrix=noise["q_matrix"],
        )
        cekf_errs.append(position_error_from_state_sequence(trajectory, x_hat_cekf[:, :, 0]))
        for scheme in diffusion_schemes:
            x_hat_dekf = diffusion_extended_kalman_filter_parallel_edge(
                measurements=measurements, f_system=diff_f_system, h_system=h_system,
                r_array=r_array, q=noise["q"], p0=p0, x0=x_init,
                j_matrix=diffusion_j_matrices[scheme],
                time_steps=num_time_steps, node_num=num_nodes,
                q_matrix=noise["q_matrix"],
            )
            scheme_errs[scheme].append(position_error_from_dekf(trajectory, x_hat_dekf))
    diffusion_errors_cekf.append(float(np.mean(cekf_errs)))
    for scheme in diffusion_schemes:
        diffusion_errors[scheme].append(float(np.mean(scheme_errs[scheme])))

diffusion_results_df = pd.DataFrame({
    "measurement_noise": measurement_noise_values,
    "CEKF (reference)": diffusion_errors_cekf,
    **{f"DEKF: {scheme}": diffusion_errors[scheme] for scheme in diffusion_schemes},
})
display(diffusion_results_df)

matched_family = ["uniform (normalized)", "metropolis", "inverse-distance"]
matched_stack = np.array([diffusion_errors[s] for s in matched_family])
all_stack = np.array([diffusion_errors[s] for s in diffusion_schemes])
shape_spread = float(np.max(np.abs(matched_stack - matched_stack.mean(0)) / matched_stack.mean(0)))
overall_spread = float(np.max(np.abs(all_stack - all_stack.mean(0)) / all_stack.mean(0)))
verdict = "negligible" if shape_spread < 0.03 else ("small" if shape_spread < 0.10 else "significant")
print(f"\nMax relative spread across matched-Q coefficient networks (shape effect): {shape_spread:.1%}")
print(f"Max relative spread across all schemes (incl. current binary):           {overall_spread:.1%}")
print(f"=> Influence of the diffusion-coefficient network on DEKF error: {verdict}")

## 7. dt-Mismatch Robustness

Fix the measurement noise and sweep the `dt` mismatch multiplier. Data always uses the true `time_delta`; each filter instead runs with `time_delta x ratio`, and the DKN checkpoint trained at each ratio is loaded (the GNN-RNN checkpoint, if present, is reused for every ratio).

In [ ]:
R_MISMATCH_EVAL = 1.0
mismatch_experiment_dir = experiment_dir
mismatch_config = config_val
if mismatch_config.get("use_dt_mismatch", False):
    dt_mismatch_values = mismatch_config.get("dt_mismatch_values", [1.0])
else:
    dt_mismatch_values = [1.0]
    print("No dt-mismatch experiment is configured; evaluating nominal dynamics only.")

print(f"Experiment:         {mismatch_experiment_dir.name}")
print(f"dt_mismatch_values: {dt_mismatch_values}")
print(f"Evaluating at r={R_MISMATCH_EVAL}")

In [ ]:
mismatch_errors_cekf = []
mismatch_errors_dekf = []
mismatch_errors_dkn = []
mismatch_errors_gnn_rnn = []

mismatch_noise = noise_for_scale(R_MISMATCH_EVAL)
r_array_mismatch = r_array_for(h_system, R_MISMATCH_EVAL)
f_data = ConstantVelocityModel(time_delta)
mismatch_gnn_rnn_process = None
mismatch_gnn_rnn_path = gnn_rnn_model_path(mismatch_experiment_dir, R_MISMATCH_EVAL)
if mismatch_gnn_rnn_path.exists():
    mismatch_gnn_rnn_process = build_gnn_rnn_model(mismatch_config)
    mismatch_gnn_rnn_process = load_state_dict_checked(mismatch_gnn_rnn_process, mismatch_gnn_rnn_path)
else:
    print(f"No GNN-RNN checkpoint found at {mismatch_gnn_rnn_path}; omitting GNN-RNN from mismatch sweep.")

for ratio in dt_mismatch_values:
    f_model_r = ConstantVelocityModel(time_delta * ratio)

    model_path = dkn_model_path(
        mismatch_experiment_dir,
        R_MISMATCH_EVAL,
        use_dt_mismatch=mismatch_config.get("use_dt_mismatch", False),
        dt_ratio=ratio,
    )
    kalman_process_m = build_dkn_model(mismatch_config, f_model_r, r_array_mismatch, x_init)
    kalman_process_m = load_state_dict_checked(kalman_process_m, model_path)

    trial_errors_cekf_m = []
    trial_errors_dekf_m = []
    trial_errors_dkn_m = []
    trial_errors_gnn_rnn_m = []

    for _ in tqdm(range(num_trials), desc=f"ratio={ratio}"):
        true_x0 = sample_initial_state(x_init, p0)
        trajectory = generate_trajectory(f_data, true_x0, num_time_steps, mismatch_noise["q_matrix"])
        measurements = generate_measurements(h_system, trajectory, r_array_mismatch)

        x_hat_cekf_m = centralized_extended_kalman_filter(
            measurements=measurements,
            f_system=f_model_r,
            h_system=h_system,
            r_array=r_array_mismatch,
            q=mismatch_noise["q"],
            p0=p0,
            x0=x_init,
            time_steps=num_time_steps,
            node_num=num_nodes,
            q_matrix=mismatch_noise["q_matrix"],
        )
        x_hat_dekf_m = diffusion_extended_kalman_filter_parallel_edge(
            measurements=measurements,
            f_system=f_model_r,
            h_system=h_system,
            r_array=r_array_mismatch,
            q=mismatch_noise["q"],
            p0=p0,
            x0=x_init,
            j_matrix=j_matrix,
            time_steps=num_time_steps,
            node_num=num_nodes,
            q_matrix=mismatch_noise["q_matrix"],
        )

        graph_data = build_graph_data_for_dkn(adjacency_matrix, h_system, trajectory, measurements)
        with torch.no_grad():
            x_hat_dkn_m = kalman_process_m(graph_data)[0].mean(dim=1)[..., 0].cpu().numpy()
        x_hat_gnn_rnn_m = predict_gnn_rnn(mismatch_gnn_rnn_process, graph_data, h_system) if mismatch_gnn_rnn_process is not None else None

        x_true = trajectory[:, 0, 0].numpy()
        y_true = trajectory[:, 2, 0].numpy()

        x_cekf = x_hat_cekf_m[:, 0, 0]
        y_cekf = x_hat_cekf_m[:, 2, 0]
        trial_errors_cekf_m.append(np.sqrt((x_true - x_cekf) ** 2 + (y_true - y_cekf) ** 2).mean())

        x_dekf = x_hat_dekf_m[:, :, 0, 0].mean(axis=1)
        y_dekf = x_hat_dekf_m[:, :, 2, 0].mean(axis=1)
        trial_errors_dekf_m.append(np.sqrt((x_true - x_dekf) ** 2 + (y_true - y_dekf) ** 2).mean())

        trial_errors_dkn_m.append(np.sqrt((x_true - x_hat_dkn_m[:, 0]) ** 2 + (y_true - x_hat_dkn_m[:, 2]) ** 2).mean())
        if x_hat_gnn_rnn_m is not None:
            trial_errors_gnn_rnn_m.append(np.sqrt((x_true - x_hat_gnn_rnn_m[:, 0]) ** 2 + (y_true - x_hat_gnn_rnn_m[:, 2]) ** 2).mean())

    mismatch_errors_cekf.append(np.mean(trial_errors_cekf_m))
    mismatch_errors_dekf.append(np.mean(trial_errors_dekf_m))
    mismatch_errors_dkn.append(np.mean(trial_errors_dkn_m))
    mismatch_errors_gnn_rnn.append(np.mean(trial_errors_gnn_rnn_m) if trial_errors_gnn_rnn_m else np.nan)

mismatch_results_df = pd.DataFrame({
    "dt_mismatch": dt_mismatch_values,
    "CEKF": mismatch_errors_cekf,
    "DEKF": mismatch_errors_dekf,
    "DKN": mismatch_errors_dkn,
    "GNN-RNN": mismatch_errors_gnn_rnn,
})

mismatch_summary_path = notebook_plot_dir / f"mismatch_robustness_r={R_MISMATCH_EVAL}_trials={num_trials}.png"
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(mismatch_results_df["dt_mismatch"], mismatch_results_df["CEKF"], "o-", linewidth=2, markersize=8, label="CEKF")
ax.plot(mismatch_results_df["dt_mismatch"], mismatch_results_df["DEKF"], "s--", linewidth=2, markersize=8, label="DEKF")
ax.plot(mismatch_results_df["dt_mismatch"], mismatch_results_df["DKN"], "^-.", linewidth=2, markersize=8, label="DKN")
if mismatch_results_df["GNN-RNN"].notna().any():
    ax.plot(mismatch_results_df["dt_mismatch"], mismatch_results_df["GNN-RNN"], "d:", linewidth=2, markersize=8, label="GNN-RNN")
ax.axvline(x=1.0, color="gray", linestyle=":", linewidth=1.5, label="No mismatch (ratio=1)")
ax.set_xlabel("dt mismatch ratio")
ax.set_ylabel("Average Position Error")
ax.set_title(f"dt Mismatch Robustness (r={R_MISMATCH_EVAL}, {num_trials} trials, mu={config_val['mu']}, T={num_time_steps})")
ax.grid(True, alpha=0.3)
ax.legend()
fig.savefig(mismatch_summary_path, dpi=200, bbox_inches="tight")
print(f"Saved: {mismatch_summary_path}")
plt.show()

mismatch_results_df

## 8. Node-Count Generalization

Evaluate the trained DKN and optional GNN-RNN against fresh scenarios with different node counts (the graph is rebuilt for each size). This checks how well the learned models transfer beyond their training graph size; CEKF and DEKF are included as references.

In [ ]:
TEST_NUM_NODES_VALUES = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]

train_num_nodes = num_nodes
NODE_COUNT_R_EVAL = measurement_noise_values[0]
NODE_COUNT_DKN_DT_RATIO = default_dkn_dt_ratio if use_dt_mismatch else None
node_count_run_name = dkn_run_name(NODE_COUNT_R_EVAL, use_dt_mismatch=use_dt_mismatch, dt_ratio=NODE_COUNT_DKN_DT_RATIO)
node_count_model_path = dkn_model_path(experiment_dir, NODE_COUNT_R_EVAL, use_dt_mismatch=use_dt_mismatch, dt_ratio=NODE_COUNT_DKN_DT_RATIO)
if not node_count_model_path.exists():
    raise FileNotFoundError(f"Missing DKN checkpoint for node-count sweep: {node_count_model_path}")

f_data_nodes = ConstantVelocityModel(time_delta)
f_model_nodes = ConstantVelocityModel(time_delta * float(NODE_COUNT_DKN_DT_RATIO)) if use_dt_mismatch else f_data_nodes
node_count_noise = noise_for_scale(NODE_COUNT_R_EVAL)

node_count_gnn_rnn_process = None
node_count_gnn_rnn_path = gnn_rnn_model_path(experiment_dir, NODE_COUNT_R_EVAL)
if node_count_gnn_rnn_path.exists():
    node_count_gnn_rnn_process = build_gnn_rnn_model(config_val)
    node_count_gnn_rnn_process = load_state_dict_checked(node_count_gnn_rnn_process, node_count_gnn_rnn_path)
else:
    print(f"No GNN-RNN checkpoint found at {node_count_gnn_rnn_path}; omitting GNN-RNN from node-count sweep.")

node_count_errors_cekf = []
node_count_errors_dekf = []
node_count_errors_dkn = []
node_count_errors_gnn_rnn = []

for test_num_nodes in TEST_NUM_NODES_VALUES:

    test_node_positions = generate_node_positions(test_num_nodes, seed=config_val["graph_seed"], area_size=config_val.get("area_size", 100.0))
    test_h_system = DistanceAngleObservation(test_node_positions)
    test_adjacency_matrix = create_distance_based_graph(
        test_node_positions,
        k_neighbors=config_val["k_neighbors"],
        seed=config_val["graph_seed"],
    )
    test_j_matrix = np.array(test_adjacency_matrix, dtype=float, copy=True)
    np.fill_diagonal(test_j_matrix, 1.0)
    test_r_array = r_array_for(test_h_system, NODE_COUNT_R_EVAL)
    node_count_kalman_process = build_dkn_model(
        config_val, f_model_nodes, test_r_array, x_init
    )
    node_count_kalman_process = load_state_dict_checked(
        node_count_kalman_process, node_count_model_path
    )

    paired_trials = []
    for _ in range(num_trials):
        true_x0 = sample_initial_state(x_init, p0)
        traj = generate_trajectory(
            f_data_nodes, true_x0, num_time_steps, node_count_noise["q_matrix"]
        )
        meas = generate_measurements(test_h_system, traj, test_r_array)
        paired_trials.append((traj, meas))

    trial_errors_cekf = []
    trial_errors_dekf = []
    trial_errors_dkn = []
    trial_errors_gnn_rnn = []

    for trajectory, measurements in tqdm(paired_trials, desc=f"nodes={test_num_nodes}"):
        x_hat_cekf_n = centralized_extended_kalman_filter(
            measurements=measurements,
            f_system=f_model_nodes,
            h_system=test_h_system,
            r_array=test_r_array,
            q=node_count_noise["q"],
            p0=p0,
            x0=x_init,
            time_steps=num_time_steps,
            node_num=test_num_nodes,
            q_matrix=node_count_noise["q_matrix"],
        )
        x_hat_dekf_n = diffusion_extended_kalman_filter_parallel_edge(
            measurements=measurements,
            f_system=f_model_nodes,
            h_system=test_h_system,
            r_array=test_r_array,
            q=node_count_noise["q"],
            p0=p0,
            x0=x_init,
            j_matrix=test_j_matrix,
            time_steps=num_time_steps,
            node_num=test_num_nodes,
            q_matrix=node_count_noise["q_matrix"],
        )

        graph_data = build_graph_data_for_dkn(test_adjacency_matrix, test_h_system, trajectory, measurements)
        with torch.no_grad():
            x_hat_dkn_n = node_count_kalman_process(graph_data)[0].mean(dim=1)[..., 0].cpu().numpy()
        x_hat_gnn_rnn_n = predict_gnn_rnn(node_count_gnn_rnn_process, graph_data, test_h_system) if node_count_gnn_rnn_process is not None else None

        x_true = trajectory[:, 0, 0].numpy()
        y_true = trajectory[:, 2, 0].numpy()

        x_cekf = x_hat_cekf_n[:, 0, 0]
        y_cekf = x_hat_cekf_n[:, 2, 0]
        trial_errors_cekf.append(np.sqrt((x_true - x_cekf) ** 2 + (y_true - y_cekf) ** 2).mean())

        x_dekf = x_hat_dekf_n[:, :, 0, 0].mean(axis=1)
        y_dekf = x_hat_dekf_n[:, :, 2, 0].mean(axis=1)
        trial_errors_dekf.append(np.sqrt((x_true - x_dekf) ** 2 + (y_true - y_dekf) ** 2).mean())

        trial_errors_dkn.append(np.sqrt((x_true - x_hat_dkn_n[:, 0]) ** 2 + (y_true - x_hat_dkn_n[:, 2]) ** 2).mean())
        if x_hat_gnn_rnn_n is not None:
            trial_errors_gnn_rnn.append(np.sqrt((x_true - x_hat_gnn_rnn_n[:, 0]) ** 2 + (y_true - x_hat_gnn_rnn_n[:, 2]) ** 2).mean())

    node_count_errors_cekf.append(np.mean(trial_errors_cekf))
    node_count_errors_dekf.append(np.mean(trial_errors_dekf))
    node_count_errors_dkn.append(np.mean(trial_errors_dkn))
    node_count_errors_gnn_rnn.append(np.mean(trial_errors_gnn_rnn) if trial_errors_gnn_rnn else np.nan)

node_count_results_df = pd.DataFrame({
    "test_num_nodes": TEST_NUM_NODES_VALUES,
    "CEKF": node_count_errors_cekf,
    "DEKF": node_count_errors_dekf,
    "DKN": node_count_errors_dkn,
    "GNN-RNN": node_count_errors_gnn_rnn,
})

node_count_summary_path = notebook_plot_dir / f"node_count_generalization_{node_count_run_name}_trainN={train_num_nodes}_trials={num_trials}.png"
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(node_count_results_df["test_num_nodes"], node_count_results_df["CEKF"], "o-", linewidth=2, markersize=8, label="CEKF")
ax.plot(node_count_results_df["test_num_nodes"], node_count_results_df["DEKF"], "s--", linewidth=2, markersize=8, label="DEKF")
ax.plot(node_count_results_df["test_num_nodes"], node_count_results_df["DKN"], "^-.", linewidth=2, markersize=8, label="DKN")
if node_count_results_df["GNN-RNN"].notna().any():
    ax.plot(node_count_results_df["test_num_nodes"], node_count_results_df["GNN-RNN"], "d:", linewidth=2, markersize=8, label="GNN-RNN")
ax.axvline(x=train_num_nodes, color="gray", linestyle=":", linewidth=1.5, label=f"Train nodes ({train_num_nodes})")
ax.set_xlabel("Number of test nodes")
ax.set_ylabel("Average Position Error")
ax.set_title(f"Node-Count Generalization,fix Graph degree=5({node_count_run_name}, {num_trials} trials, T={num_time_steps})")
ax.grid(True, alpha=0.3)
ax.legend()
fig.savefig(node_count_summary_path, dpi=200, bbox_inches="tight")
print(f"Saved: {node_count_summary_path}")
plt.show()

node_count_results_df


## 9. Time-Step Generalization

Evaluate the trained models on shorter, training-length, and longer trajectories without retraining. Every model receives the same paired trials for each horizon, and the vertical reference line marks the number of time steps used during training.

In [ ]:
TIME_STEPS_R_EVAL = measurement_noise_values[0]
TIME_STEPS_VALUES = sorted({
    max(2, num_time_steps // 2),
    num_time_steps,
    num_time_steps * 2,
})
TIME_STEPS_DKN_DT_RATIO = default_dkn_dt_ratio if use_dt_mismatch else 1.0

steps_noise = noise_for_scale(TIME_STEPS_R_EVAL)
steps_r_array = r_array_for(h_system, TIME_STEPS_R_EVAL)
steps_data_system = ConstantVelocityModel(time_delta)
steps_model_system = ConstantVelocityModel(
    time_delta * float(TIME_STEPS_DKN_DT_RATIO)
)

steps_dkn_path = dkn_model_path(
    experiment_dir,
    TIME_STEPS_R_EVAL,
    use_dt_mismatch=use_dt_mismatch,
    dt_ratio=TIME_STEPS_DKN_DT_RATIO,
)
if not steps_dkn_path.exists():
    raise FileNotFoundError(
        f"Missing DKN checkpoint for time-step generalization: {steps_dkn_path}"
    )
steps_dkn = build_dkn_model(
    config_val, steps_model_system, steps_r_array, x_init
)
steps_dkn = load_state_dict_checked(steps_dkn, steps_dkn_path).eval()

steps_gnn_rnn = None
steps_gnn_path = gnn_rnn_model_path(experiment_dir, TIME_STEPS_R_EVAL)
if steps_gnn_path.exists():
    steps_gnn_rnn = build_gnn_rnn_model(config_val)
    steps_gnn_rnn = load_state_dict_checked(steps_gnn_rnn, steps_gnn_path).eval()
else:
    print(
        f"No GNN-RNN checkpoint found at {steps_gnn_path}; "
        "omitting it from time-step generalization."
    )

steps_model_names = ["CEKF", "DEKF", "DKN"]
if steps_gnn_rnn is not None:
    steps_model_names.append("GNN-RNN")
steps_rows = []
longest_steps_prediction = None

for test_time_steps in TIME_STEPS_VALUES:
    seed_everything(config_val["seed"] + 70_000)
    trial_errors = {name: [] for name in steps_model_names}
    for trial_idx in tqdm(
        range(num_trials), desc=f"time-step generalization T={test_time_steps}"
    ):
        true_x0 = sample_initial_state(x_init, p0)
        trajectory, measurements = generate_trial_data(
            steps_data_system,
            h_system,
            true_x0,
            test_time_steps,
            steps_noise["q_matrix"],
            steps_r_array,
        )
        x_hat_cekf = centralized_extended_kalman_filter(
            measurements=measurements,
            f_system=steps_model_system,
            h_system=h_system,
            r_array=steps_r_array,
            q=steps_noise["q"],
            p0=p0,
            x0=x_init,
            time_steps=test_time_steps,
            node_num=num_nodes,
            q_matrix=steps_noise["q_matrix"],
        )
        x_hat_dekf = diffusion_extended_kalman_filter_parallel_edge(
            measurements=measurements,
            f_system=steps_model_system,
            h_system=h_system,
            r_array=steps_r_array,
            q=steps_noise["q"],
            p0=p0,
            x0=x_init,
            j_matrix=j_matrix,
            time_steps=test_time_steps,
            node_num=num_nodes,
            q_matrix=steps_noise["q_matrix"],
        )
        graph_data = build_graph_data_for_dkn(
            adjacency_matrix, h_system, trajectory, measurements
        )
        with torch.no_grad():
            x_hat_dkn = steps_dkn(graph_data)[0].mean(dim=1)[..., 0].cpu().numpy()
        trial_errors["CEKF"].append(
            position_error_from_state_sequence(trajectory, x_hat_cekf[:, :, 0])
        )
        trial_errors["DEKF"].append(
            position_error_from_dekf(trajectory, x_hat_dekf)
        )
        trial_errors["DKN"].append(
            position_error_from_state_sequence(trajectory, x_hat_dkn)
        )
        x_hat_gnn_rnn = None
        if steps_gnn_rnn is not None:
            x_hat_gnn_rnn = predict_gnn_rnn(
                steps_gnn_rnn, graph_data, h_system
            )
            trial_errors["GNN-RNN"].append(
                position_error_from_state_sequence(trajectory, x_hat_gnn_rnn)
            )

        if test_time_steps == max(TIME_STEPS_VALUES) and trial_idx == 0:
            longest_steps_prediction = {
                "trajectory": trajectory,
                "cekf": x_hat_cekf,
                "dekf": x_hat_dekf,
                "dkn": x_hat_dkn,
                "gnn_rnn": x_hat_gnn_rnn,
            }

    for model_name, errors in trial_errors.items():
        errors = np.asarray(errors)
        steps_rows.append({
            "time_steps": test_time_steps,
            "model": model_name,
            "mean_position_error": errors.mean(),
            "std_position_error": errors.std(ddof=1),
            "ci95": 1.96 * errors.std(ddof=1) / np.sqrt(len(errors)),
        })

if longest_steps_prediction is None:
    raise RuntimeError("No longest-horizon prediction sample was captured.")
longest_time_steps = max(TIME_STEPS_VALUES)
longest_tracking_plot_path = notebook_plot_dir / (
    f"time_steps_longest_prediction_T={longest_time_steps}.png"
)
plot_tracking_results(
    longest_steps_prediction["trajectory"],
    longest_steps_prediction["cekf"],
    x_hat_dekf=longest_steps_prediction["dekf"],
    x_hat_dkn=longest_steps_prediction["dkn"],
    x_hat_gnn_rnn=longest_steps_prediction["gnn_rnn"],
    node_positions=node_positions,
    node_types=node_types,
    save_path=longest_tracking_plot_path,
)
print(f"Saved longest-horizon tracking plot: {longest_tracking_plot_path}")

time_steps_results_df = pd.DataFrame(steps_rows)
epsilon = np.finfo(float).tiny
time_steps_results_df["mean_position_error_db"] = 10 * np.log10(
    np.maximum(time_steps_results_df["mean_position_error"], epsilon)
)
time_steps_results_df["ci95_low_db"] = 10 * np.log10(
    np.maximum(
        time_steps_results_df["mean_position_error"] - time_steps_results_df["ci95"],
        epsilon,
    )
)
time_steps_results_df["ci95_high_db"] = 10 * np.log10(
    np.maximum(
        time_steps_results_df["mean_position_error"] + time_steps_results_df["ci95"],
        epsilon,
    )
)
display(time_steps_results_df)

time_steps_plot_path = notebook_plot_dir / (
    f"time_steps_generalization_trainT={num_time_steps}_trials={num_trials}.png"
)
plt.figure(figsize=(10, 6))
for model_name, rows in time_steps_results_df.groupby("model"):
    rows = rows.sort_values("time_steps")
    mean_db = rows["mean_position_error_db"].to_numpy()
    error_db = np.vstack([
        mean_db - rows["ci95_low_db"].to_numpy(),
        rows["ci95_high_db"].to_numpy() - mean_db,
    ])
    plt.errorbar(
        rows["time_steps"],
        mean_db,
        yerr=error_db,
        marker="o",
        capsize=4,
        linewidth=2,
        label=model_name,
    )
plt.axvline(
    num_time_steps,
    color="black",
    linestyle="--",
    linewidth=1,
    label=f"training T={num_time_steps}",
)
plt.xlabel("Trajectory time steps")
plt.ylabel("Mean position error [dB]")
plt.title(
    f"Time-Step Generalization ({num_trials} paired trials, r={TIME_STEPS_R_EVAL:g})"
)
plt.grid(True, alpha=0.3)
plt.legend()
plt.savefig(time_steps_plot_path, dpi=200, bbox_inches="tight")
print(f"Saved: {time_steps_plot_path}")
plt.show()

## 10. Inference-Latency Comparison

Measure wall-clock inference time per trial for CEKF, DEKF, and DKN on the current device. DKN uses batched inference via `Batch.from_data_list()`; the classical filters are inherently sequential. The bar chart shows mean per-trial latency +/- one standard deviation.

In [ ]:
import time
from torch_geometric.data import Batch


LATENCY_EXP_DIR = experiment_dir
LATENCY_R = measurement_noise_values[0]
LATENCY_DT_RATIO = default_dkn_dt_ratio if use_dt_mismatch else 1.0
LATENCY_NUM_TRIALS = 200
LATENCY_BATCH_SIZE = 50
LATENCY_WARMUP = 5

lat_cfg = load_experiment_config(LATENCY_EXP_DIR)
lat_state_dim  = lat_cfg["state_dimension"]
lat_x_init     = np.array(lat_cfg["x0"], dtype=float).reshape(lat_state_dim, 1)
lat_p0         = np.eye(lat_state_dim) * lat_cfg["p0_scale"]
lat_num_nodes  = lat_cfg["num_nodes"]
lat_time_delta = lat_cfg["time_delta"]
lat_noise      = derive_localization_noise(lat_cfg["mu"], lat_cfg["rho"], LATENCY_R)
lat_T          = experiment_time_steps(lat_cfg)

lat_node_positions = np.array(lat_cfg["node_positions"], dtype=float)
lat_h_system = DistanceAngleObservation(lat_node_positions)
lat_adj = create_distance_based_graph(
    lat_node_positions, k_neighbors=lat_cfg["k_neighbors"], seed=lat_cfg["graph_seed"]
)
lat_j_matrix = np.array(lat_adj, dtype=float, copy=True)
np.fill_diagonal(lat_j_matrix, 1.0)
lat_r_array = measurement_noise_for_nodes(
    lat_h_system, lat_noise["sigma_r"], lat_noise["sigma_theta"]
)
lat_f_system = ConstantVelocityModel(lat_time_delta * LATENCY_DT_RATIO)


lat_dkn_path = dkn_model_path(
    LATENCY_EXP_DIR,
    LATENCY_R,
    use_dt_mismatch=lat_cfg.get("use_dt_mismatch", False),
    dt_ratio=LATENCY_DT_RATIO,
)
if not lat_dkn_path.exists():
    raise FileNotFoundError(f"Missing DKN checkpoint: {lat_dkn_path}")

lat_dkn = build_dkn_model(lat_cfg, lat_f_system, lat_r_array, lat_x_init)
lat_dkn = load_state_dict_checked(lat_dkn, lat_dkn_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lat_dkn = lat_dkn.to(device)
print(f"DKN device: {device}")


seed_everything(lat_cfg["seed"])
total_trials = LATENCY_NUM_TRIALS + LATENCY_WARMUP * LATENCY_BATCH_SIZE
lat_trials = []
for _ in range(total_trials):
    true_x0 = sample_initial_state(lat_x_init, lat_p0)
    traj = generate_trajectory(lat_f_system, true_x0, lat_T, lat_noise["q_matrix"])
    meas = generate_measurements(lat_h_system, traj, lat_r_array)
    lat_trials.append((traj, meas))


lat_data_list = [
    build_graph_data_for_dkn(lat_adj, lat_h_system, t, m) for t, m in lat_trials
]

for wi in range(LATENCY_WARMUP):
    warm_batch = Batch.from_data_list(
        lat_data_list[wi * LATENCY_BATCH_SIZE : (wi + 1) * LATENCY_BATCH_SIZE]
    ).to(device)
    with torch.no_grad():
        _ = lat_dkn(warm_batch)
sync_torch_device(device)

timing_data   = lat_data_list[LATENCY_WARMUP * LATENCY_BATCH_SIZE:]
timing_trials = lat_trials[LATENCY_WARMUP * LATENCY_BATCH_SIZE:]


cekf_times = []
for traj, meas in tqdm(timing_trials, desc="Timing CEKF"):
    t0 = time.perf_counter()
    _ = centralized_extended_kalman_filter(
        measurements=meas, f_system=lat_f_system, h_system=lat_h_system,
        r_array=lat_r_array, q=lat_noise["q"], p0=lat_p0, x0=lat_x_init,
        time_steps=lat_T, node_num=lat_num_nodes,
        q_matrix=lat_noise["q_matrix"],
    )
    cekf_times.append(time.perf_counter() - t0)


dekf_times = []
for traj, meas in tqdm(timing_trials, desc="Timing DEKF"):
    t0 = time.perf_counter()
    _ = diffusion_extended_kalman_filter_parallel_edge(
        measurements=meas, f_system=lat_f_system, h_system=lat_h_system,
        r_array=lat_r_array, q=lat_noise["q"], p0=lat_p0, x0=lat_x_init,
        j_matrix=lat_j_matrix, time_steps=lat_T, node_num=lat_num_nodes,
        q_matrix=lat_noise["q_matrix"],
    )
    dekf_times.append(time.perf_counter() - t0)


dkn_per_trial_ms = []
for bi in tqdm(range(0, len(timing_data), LATENCY_BATCH_SIZE), desc="Timing DKN (batch)"):
    chunk = timing_data[bi : bi + LATENCY_BATCH_SIZE]
    if not chunk:
        break
    batch = Batch.from_data_list(chunk).to(device)
    sync_torch_device(device)
    t0 = time.perf_counter()
    with torch.no_grad():
        _ = lat_dkn(batch)
    sync_torch_device(device)
    dkn_per_trial_ms.extend([1e3 * (time.perf_counter() - t0) / len(chunk)] * len(chunk))


results_latency = {
    "CEKF":                             np.array(cekf_times) * 1e3,
    "DEKF":                             np.array(dekf_times) * 1e3,
    f"DKN (batch={LATENCY_BATCH_SIZE})": np.array(dkn_per_trial_ms),
}

latency_df = pd.DataFrame({
    label: {"mean_ms": v.mean(), "std_ms": v.std(), "median_ms": np.median(v)}
    for label, v in results_latency.items()
}).T
print(f"\nInference latency per trial (ms) - device: {device}")
print(latency_df.round(3).to_string())


fig, ax = plt.subplots(figsize=(8, 5))
labels = list(results_latency.keys())
means  = [results_latency[l].mean() for l in labels]
stds   = [results_latency[l].std()  for l in labels]
colors = ["#4C72B0", "#DD8452", "#55A868"]
bars = ax.bar(labels, means, yerr=stds, capsize=6, color=colors, edgecolor="black", linewidth=0.8)
for bar, m, s in zip(bars, means, stds):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        m + s + max(means) * 0.015,
        f"{m:.2f} ms",
        ha="center", va="bottom", fontsize=10,
    )
ax.set_ylabel("Mean per-trial latency (ms)")
ax.set_title(
    f"Inference Latency Comparison\n"
    f"r={LATENCY_R}, dtx{LATENCY_DT_RATIO}, T={lat_T}, "
    f"nodes={lat_num_nodes}, device={device}, N={LATENCY_NUM_TRIALS} trials"
)
ax.grid(axis="y", alpha=0.35)
ax.set_ylim(0, max(means) * 1.35)

lat_plot_path = notebook_plot_dir / (
    f"inference_latency_r={LATENCY_R}_dtx{LATENCY_DT_RATIO}_batch={LATENCY_BATCH_SIZE}.png"
)
fig.savefig(lat_plot_path, dpi=200, bbox_inches="tight")
print(f"Saved: {lat_plot_path}")
plt.show()

latency_df


## 11. Detailed Multi-Model Comparison

Choose any two or more localization experiments in `MODEL_EXPERIMENTS`. The checkpoints are validated against one shared scenario and evaluated on identical paired trials under the initial-state uncertainty read from `run.log`. The analysis reports model configuration, ranking, mean/standard-deviation/median error, and paired model-to-model differences.

In [ ]:
MODEL_EXPERIMENTS = [4, 5]  # Edit this list; at least two IDs are required.
COMPARISON_TRIALS = None  # None uses num_trials from the reference experiment.

if len(set(MODEL_EXPERIMENTS)) < 2:
    raise ValueError("MODEL_EXPERIMENTS must contain at least two unique experiment IDs.")

model_dirs = [save_root / f"experiment_{idx}" for idx in MODEL_EXPERIMENTS]
model_configs = [load_experiment_config(path) for path in model_dirs]
reference_config = model_configs[0]
shared_fields = (
    "state_dimension", "area_size", "x0", "time_delta", "mu", "rho", "r_scale",
    "p0_scale", "num_nodes", "graph_seed", "k_neighbors", "time_steps",
)
for path, config in zip(model_dirs[1:], model_configs[1:]):
    differences = [key for key in shared_fields if config[key] != reference_config[key]]
    if differences:
        raise ValueError(f"{path.name} differs in shared scenario fields: {differences}")

TEST_P0_REGIMES = [float(reference_config["p0_scale"])]
print(f"Using p0_scale={TEST_P0_REGIMES[0]:g} from {model_dirs[0] / 'run.log'}")
comparison_r_scale = reference_config["r_scale"][0]
comparison_noise = derive_localization_noise(
    reference_config["mu"], reference_config["rho"], comparison_r_scale
)
nominal_x0 = np.asarray(reference_config["x0"], dtype=float).reshape(
    reference_config["state_dimension"], 1
)
comparison_data_system = ConstantVelocityModel(reference_config["time_delta"])
comparison_positions = np.asarray(reference_config["node_positions"], dtype=float)
comparison_h_system = DistanceAngleObservation(comparison_positions)
comparison_adjacency = create_distance_based_graph(
    comparison_positions,
    k_neighbors=reference_config["k_neighbors"],
    seed=reference_config["graph_seed"],
)
comparison_r_array = measurement_noise_for_nodes(
    comparison_h_system,
    comparison_noise["sigma_r"],
    comparison_noise["sigma_theta"],
)

model_rows = []
comparison_models = {}
for path, config in zip(model_dirs, model_configs):
    uses_mismatch = config.get("use_dt_mismatch", False)
    model_dt_ratio = (
        nearest_nominal_dt_ratio(config.get("dt_mismatch_values", [1.0]))
        if uses_mismatch
        else 1.0
    )
    checkpoint = dkn_model_path(
        path,
        comparison_r_scale,
        use_dt_mismatch=uses_mismatch,
        dt_ratio=model_dt_ratio,
        config=config,
    )
    status = "missing checkpoint"
    nonfinite_parameters = np.nan
    if checkpoint.exists():
        checkpoint_data = torch.load(checkpoint, map_location="cpu")
        state_dict = checkpoint_data.get("state_dict", checkpoint_data)
        nonfinite_parameters = sum(
            int((~torch.isfinite(value)).sum())
            for value in state_dict.values()
            if torch.is_tensor(value) and value.is_floating_point()
        )
        if nonfinite_parameters == 0:
            model_system = ConstantVelocityModel(
                config["time_delta"] * model_dt_ratio
            )
            model = build_dkn_model(
                config, model_system, comparison_r_array, nominal_x0
            )
            comparison_models[path.name] = load_state_dict_checked(
                model, checkpoint
            ).eval()
            status = "valid"
        else:
            status = "non-finite checkpoint"
    model_rows.append({
        "experiment": path.name,
        "description": config.get("description", ""),
        "dt_ratio": model_dt_ratio,
        "checkpoint": str(checkpoint),
        "train_p0": config["p0_scale"],
        "position_only_loss": config["position_only_loss"],
        "learn_edge_kalman": config["learn_edge_kalman"],
        "consensus_layer": config["consensus_layer"],
        "status": status,
        "nonfinite_parameters": nonfinite_parameters,
    })

model_config_df = pd.DataFrame(model_rows)
display(model_config_df)
if len(comparison_models) < 2:
    raise RuntimeError("At least two selected experiments must have valid checkpoints.")

comparison_trials = int(COMPARISON_TRIALS or reference_config["num_trials"])
error_rows = []
for regime_idx, test_p0 in enumerate(TEST_P0_REGIMES):
    initial_rng = np.random.default_rng(
        reference_config["seed"] + 20_000 + regime_idx
    )
    seed_everything(reference_config["seed"] + 30_000 + regime_idx)
    for trial_idx in tqdm(
        range(comparison_trials), desc=f"Model comparison: test p0={test_p0:g}"
    ):
        true_x0 = nominal_x0.copy()
        if test_p0:
            true_x0 += np.sqrt(test_p0) * initial_rng.standard_normal(true_x0.shape)
        trajectory, measurements = generate_trial_data(
            comparison_data_system,
            comparison_h_system,
            true_x0,
            reference_config["time_steps"],
            comparison_noise["q_matrix"],
            comparison_r_array,
        )
        graph_data = build_graph_data_for_dkn(
            comparison_adjacency, comparison_h_system, trajectory, measurements
        )
        with torch.no_grad():
            for experiment, model in comparison_models.items():
                estimate = model(graph_data)[0].mean(dim=1)[..., 0].cpu().numpy()
                error_rows.append({
                    "test_p0": test_p0,
                    "trial": trial_idx,
                    "experiment": experiment,
                    "position_error": position_error_from_state_sequence(
                        trajectory, estimate
                    ),
                })

comparison_raw = pd.DataFrame(error_rows)
comparison_summary = (
    comparison_raw.groupby(["test_p0", "experiment"])["position_error"]
    .agg(["mean", "std", "median", "count"])
    .reset_index()
)
comparison_summary["ci95"] = (
    1.96 * comparison_summary["std"] / np.sqrt(comparison_summary["count"])
)
comparison_summary["rank"] = comparison_summary.groupby("test_p0")["mean"].rank(
    method="min"
)
regime_best = comparison_summary.groupby("test_p0")["mean"].transform("min")
comparison_summary["normalized_error"] = comparison_summary["mean"] / regime_best
overall_ranking = (
    comparison_summary.groupby("experiment")["normalized_error"]
    .mean()
    .sort_values()
    .rename("overall_score")
    .reset_index()
)

paired = comparison_raw.pivot(
    index=["test_p0", "trial"], columns="experiment", values="position_error"
)
paired_rows = []
model_names = list(paired.columns)
for test_p0, regime_paired in paired.groupby(level="test_p0"):
    for left_idx, left in enumerate(model_names):
        for right in model_names[left_idx + 1:]:
            delta = regime_paired[left] - regime_paired[right]
            paired_rows.append({
                "test_p0": test_p0,
                "model_a": left,
                "model_b": right,
                "mean_error_a_minus_b": delta.mean(),
                "median_error_a_minus_b": delta.median(),
                "model_a_win_rate": float((delta < 0).mean()),
            })
paired_comparison = pd.DataFrame(paired_rows)

display(comparison_summary.sort_values(["test_p0", "rank"]))
display(overall_ranking)
display(paired_comparison)

comparison_output_dir = save_root / "comparison_selected_models"
comparison_output_dir.mkdir(parents=True, exist_ok=True)
comparison_raw.to_csv(comparison_output_dir / "per_trial_errors.csv", index=False)
comparison_summary.to_csv(comparison_output_dir / "summary.csv", index=False)
overall_ranking.to_csv(comparison_output_dir / "overall_ranking.csv", index=False)
paired_comparison.to_csv(comparison_output_dir / "paired_differences.csv", index=False)
model_config_df.to_csv(comparison_output_dir / "model_configs.csv", index=False)

fig, axis = plt.subplots(figsize=(12, 6))
plot_summary = comparison_summary.sort_values(["test_p0", "experiment"])
regimes = list(plot_summary["test_p0"].unique())
experiments = list(comparison_models)
x_positions = np.arange(len(experiments))
bar_width = 0.8 / len(regimes)
for regime_idx, test_p0 in enumerate(regimes):
    regime_rows = plot_summary.set_index(["test_p0", "experiment"]).loc[test_p0]
    means = [regime_rows.loc[name, "mean"] for name in experiments]
    ci95 = [regime_rows.loc[name, "ci95"] for name in experiments]
    offset = (regime_idx - (len(regimes) - 1) / 2) * bar_width
    axis.bar(
        x_positions + offset,
        means,
        width=bar_width,
        yerr=ci95,
        capsize=4,
        label=f"test p0={test_p0:g}",
    )
axis.set_xticks(x_positions, experiments, rotation=25, ha="right")
axis.set_ylabel("Mean position error")
axis.set_title(f"Selected Models on {comparison_trials} Paired Trials per Regime")
axis.grid(axis="y", alpha=0.3)
axis.legend()
fig.tight_layout()
comparison_plot_path = comparison_output_dir / "position_error_by_regime.png"
fig.savefig(comparison_plot_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved detailed comparison to: {comparison_output_dir}")

## 12. r=1.0 DKN Comparison: experiment_15 vs experiment_9

Compare the four DKN checkpoints trained at **r=1.0** - the two runs in
`experiment_15` and the two DKN runs in `experiment_9` (with and without the
`dt` mismatch). `experiment_9` and `experiment_15` were trained on the identical
scenario, so every model is evaluated on **one shared set of paired Monte-Carlo
trials**. The metric is the same one used by the noise-sweep plots:
`10 * log10` of the mean Euclidean position error, reported here as **MSE [dB]**
(lower is better).

In [ ]:
# r=1.0 DKN comparison: experiment_15 (two runs) vs experiment_9 (two DKN runs).
# Same metric as the noise-sweep plots: 10*log10 of the mean Euclidean position
# error, averaged over paired trials. Reported as "MSE [dB]" (lower is better).
COMPARE_R = 1.0
COMPARE_MODELS = [
    ("experiment_15", 1.0, "exp15 DKN r=1.0 (no mismatch, dtx1)"),
    ("experiment_15", 2.0, "exp15 DKN r=1.0 (mismatch, dtx2)"),
    ("experiment_9",  1.0, "exp9 DKN r=1.0 (no mismatch, dtx1)"),
    ("experiment_9",  2.0, "exp9 DKN r=1.0 (mismatch, dtx2)"),
]

# Shared scenario (exp_9 and exp_15 were trained on identical settings).
ref_dir = save_root / COMPARE_MODELS[0][0]
ref_cfg = load_experiment_config(ref_dir)
cmp_state_dim = ref_cfg["state_dimension"]
cmp_x_init = np.array(ref_cfg["x0"], dtype=float).reshape(cmp_state_dim, 1)
cmp_p0 = np.eye(cmp_state_dim) * ref_cfg["p0_scale"]
cmp_T = experiment_time_steps(ref_cfg)
cmp_time_delta = ref_cfg["time_delta"]
cmp_trials = ref_cfg["num_trials"]
cmp_positions = np.array(ref_cfg["node_positions"], dtype=float)
cmp_h_system = DistanceAngleObservation(cmp_positions)
cmp_adjacency = create_distance_based_graph(
    cmp_positions, k_neighbors=ref_cfg["k_neighbors"], seed=ref_cfg["graph_seed"]
)
cmp_noise = derive_localization_noise(ref_cfg["mu"], ref_cfg["rho"], COMPARE_R)
cmp_r_array = measurement_noise_for_nodes(
    cmp_h_system, cmp_noise["sigma_r"], cmp_noise["sigma_theta"]
)
cmp_f_data = ConstantVelocityModel(cmp_time_delta)  # true dynamics generate the data

# Load the four DKN checkpoints (each built with its own training config).
cmp_processes = []
for exp_name, dt_ratio, label in COMPARE_MODELS:
    exp_dir = save_root / exp_name
    exp_cfg = load_experiment_config(exp_dir)
    path = dkn_model_path(exp_dir, COMPARE_R, use_dt_mismatch=True, dt_ratio=dt_ratio)
    if not path.exists():
        raise FileNotFoundError(f"Missing DKN checkpoint: {path}")
    f_model = ConstantVelocityModel(cmp_time_delta * float(dt_ratio))
    model = build_dkn_model(exp_cfg, f_model, cmp_r_array, cmp_x_init)
    model = load_state_dict_checked(model, path).eval()
    cmp_processes.append((label, model))

# Paired Monte-Carlo evaluation: one shared trajectory/measurement set per trial.
seed_everything(ref_cfg["seed"])
cmp_errors = {label: [] for label, _ in cmp_processes}
for _ in tqdm(range(cmp_trials), desc=f"r={COMPARE_R} comparison"):
    true_x0 = np.random.multivariate_normal(cmp_x_init[:, 0], cmp_p0).reshape(cmp_x_init.shape)
    trajectory, measurements = generate_trial_data(
        cmp_f_data, cmp_h_system, true_x0, cmp_T, cmp_noise["q_matrix"], cmp_r_array
    )
    graph_data = build_graph_data_for_dkn(cmp_adjacency, cmp_h_system, trajectory, measurements)
    for label, model in cmp_processes:
        with torch.no_grad():
            x_hat = model(graph_data)[0].mean(dim=1)[..., 0].cpu().numpy()
        cmp_errors[label].append(position_error_from_state_sequence(trajectory, x_hat))

comparison_r1_table = pd.DataFrame(
    {
        "Model": [label for label, _ in cmp_processes],
        "MSE [dB]": [10 * np.log10(np.mean(cmp_errors[label])) for label, _ in cmp_processes],
    }
)
comparison_r1_table

## 13. Consensus Layer Ablation: Simple vs Adaptive

Compares the two consensus variants on the paired trials produced by section 11. Run section 11 first with `MODEL_EXPERIMENTS` set to one `consensus=simple` and one `consensus=adaptive` experiment that share the same scenario. This cell reads the saved CSVs, so the figure can be restyled without repeating the trial sweep.


In [ ]:
from scipy.stats import wilcoxon

CONSENSUS_DIR = save_root / "comparison_selected_models"
CONSENSUS_BASELINE = "simple"  # reference variant; the other layer is the contender

consensus_configs = pd.read_csv(CONSENSUS_DIR / "model_configs.csv")
consensus_trials = pd.read_csv(CONSENSUS_DIR / "per_trial_errors.csv")

valid_configs = consensus_configs[consensus_configs["status"] == "valid"]
layer_of = dict(zip(valid_configs["experiment"], valid_configs["consensus_layer"]))
if len(layer_of) != 2 or len(set(layer_of.values())) != 2:
    raise RuntimeError(
        "This figure expects exactly two valid checkpoints with different "
        f"consensus_layer values; got {layer_of}. Re-run section 11 with a "
        "consensus-layer pair in MODEL_EXPERIMENTS."
    )
if CONSENSUS_BASELINE not in set(layer_of.values()):
    raise RuntimeError(
        f"No experiment uses consensus_layer={CONSENSUS_BASELINE!r}; got {layer_of}."
    )
if consensus_trials["test_p0"].nunique() != 1:
    raise RuntimeError("Expected a single test_p0 regime in per_trial_errors.csv.")

baseline_exp = next(e for e, layer in layer_of.items() if layer == CONSENSUS_BASELINE)
variant_exp = next(e for e in layer_of if e != baseline_exp)
baseline_label = layer_of[baseline_exp]
variant_label = layer_of[variant_exp]

paired_errors = consensus_trials.pivot(
    index="trial", columns="experiment", values="position_error"
)
if paired_errors[[baseline_exp, variant_exp]].isna().any().any():
    raise RuntimeError("per_trial_errors.csv is not fully paired across experiments.")
baseline_errors = paired_errors[baseline_exp].to_numpy()
variant_errors = paired_errors[variant_exp].to_numpy()
delta = baseline_errors - variant_errors  # positive means the variant wins


def _ci95(values):
    return 1.96 * values.std(ddof=1) / np.sqrt(values.size)


num_paired_trials = delta.size
variant_win_rate = float((delta > 0).mean())
relative_gain = 100.0 * delta.mean() / baseline_errors.mean()
wilcoxon_p = float(wilcoxon(baseline_errors, variant_errors).pvalue)

consensus_ref_config = load_experiment_config(save_root / baseline_exp)
consensus_r_scale = consensus_ref_config["r_scale"][0]
consensus_time_steps = experiment_time_steps(consensus_ref_config)
consensus_test_p0 = float(consensus_trials["test_p0"].iloc[0])

baseline_color = "#4C72B0"
variant_color = "#DD8452"
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

means = [baseline_errors.mean(), variant_errors.mean()]
error_bars = [_ci95(baseline_errors), _ci95(variant_errors)]
bars = axes[0].bar(
    [0, 1], means, yerr=error_bars, capsize=6, width=0.55,
    color=[baseline_color, variant_color], edgecolor="black", linewidth=0.6,
)
for bar, mean_value, error_bar in zip(bars, means, error_bars):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2, mean_value + error_bar + 0.02,
        f"{mean_value:.3f}", ha="center", va="bottom", fontweight="bold",
    )
axes[0].set_xticks(
    [0, 1],
    [f"{baseline_label}\n({baseline_exp})", f"{variant_label}\n({variant_exp})"],
)
axes[0].set_ylabel("Mean position error")
axes[0].set_title(
    f"Mean error with 95% CI\n{variant_label} is {relative_gain:.1f}% lower"
)
axes[0].set_ylim(0, max(means) * 1.3)
axes[0].grid(axis="y", alpha=0.3)
axes[0].set_axisbelow(True)

limits = [
    min(baseline_errors.min(), variant_errors.min()) * 0.94,
    max(baseline_errors.max(), variant_errors.max()) * 1.06,
]
axes[1].scatter(
    baseline_errors, variant_errors, s=38, color=variant_color,
    edgecolor="black", linewidth=0.4, alpha=0.85, zorder=3,
)
axes[1].plot(limits, limits, "k--", linewidth=1.2, label="equal error", zorder=2)
axes[1].set_xlim(limits)
axes[1].set_ylim(limits)
axes[1].set_aspect("equal")
axes[1].set_xlabel(f"{baseline_label} consensus error")
axes[1].set_ylabel(f"{variant_label} consensus error")
axes[1].set_title(
    f"Paired trials (n={num_paired_trials})\n"
    f"{variant_label} wins {variant_win_rate:.0%} (below the line)"
)
axes[1].grid(alpha=0.3)
axes[1].set_axisbelow(True)
axes[1].legend(loc="upper left")

axes[2].hist(
    delta, bins=14, color=variant_color, edgecolor="black", linewidth=0.6, alpha=0.9,
)
axes[2].axvline(0, color="black", linestyle="--", linewidth=1.2, label="no difference")
axes[2].axvline(
    delta.mean(), color="#C44E52", linewidth=2, label=f"mean {delta.mean():+.3f}",
)
axes[2].set_xlabel(f"{baseline_label} error - {variant_label} error")
axes[2].set_ylabel("Trials")
axes[2].set_title(f"Paired difference\nWilcoxon signed-rank p={wilcoxon_p:.2g}")
axes[2].grid(axis="y", alpha=0.3)
axes[2].set_axisbelow(True)
axes[2].legend()

fig.suptitle(
    f"Consensus layer ablation: {baseline_label} vs {variant_label} "
    f"(r={consensus_r_scale:g}, T={consensus_time_steps}, "
    f"test p0={consensus_test_p0:g})",
    fontsize=13,
)
fig.tight_layout(rect=(0, 0, 1, 0.92))
consensus_figure_path = (
    CONSENSUS_DIR / f"consensus_{baseline_label}_vs_{variant_label}.png"
)
fig.savefig(consensus_figure_path, dpi=200, bbox_inches="tight")
plt.show()
print(
    f"{variant_label} vs {baseline_label}: mean {variant_errors.mean():.4f} vs "
    f"{baseline_errors.mean():.4f} ({relative_gain:.1f}% lower), win rate "
    f"{variant_win_rate:.0%}, Wilcoxon p={wilcoxon_p:.3g}"
)
print(f"Saved figure: {consensus_figure_path}")
